# BioJEPA v0.6 Hyperparameter Optimization

Two-phase Bayesian HPO for pretraining hyperparameters:
- **Phase 1**: Exploration on balanced 50% data subsample (15 trials)
- **Phase 2**: Refinement on full data, warm-started from Phase 1 top configs (25 trials)

See `bayes_opt.md` for design rationale.

In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from functools import partial
import numpy as np
import gc
import torch
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

from biojepa_v0_6 import BioJepaConfig
from training_v0_6 import create_model
from dataloader_v0_6 import PretrainLoader
from evals.evals import EvalContext
from bayesian_optimization.hpo_utils import (
    is_valid_config, compute_step_budget, derive_parameters,
    check_identity_shortcut_cheap, check_identity_shortcut_full,
    check_cell_type_collapse, check_vicreg_collapse, check_perturbation_signal,
    compute_objective, create_phase1_subsample,
    run_selected_evals, SubsampledPretrainLoader
)

hpo_config = {
    # Study settings
    'study_name': 'biojepa_v0_6_hpo',
    'seed': 42,

    # Hardware
    'batch_size': 64,

    # Sampler settings
    'n_startup_trials': 10,
    'multivariate': True,
    'constant_liar': True,

    # Pruner settings
    'pruner_startup_trials': 10,
    'pruner_warmup_steps': 5,

    # Phase 1 subsampling
    'phase1_fraction': 0.5,
    'phase1_balance': 'equal',

    # Eval intervals (steps)
    'cheap_eval_interval': 2500,
    'full_eval_interval': 7500,

    # Search space bounds
    'embed_dim_options': [256, 384, 512, 768],
    'n_layer_options': [4, 6, 8, 12],
    'heads_options': [4, 8, 12, 16],
    'n_pre_layer_ratio_range': (0.25, 0.75),
    'mask_ratio_range': (0.4, 0.85),
    'gaussian_scale_range': (1.0, 4.0),
    'ema_momentum_range': (0.985, 0.9995),
    'vicreg_weight_range': (0.5, 2.0),
    'film_linear_multiple_range': (0.5, 2.0),
    'lr_range': (5e-5, 3e-3),

    # Fixed params
    'num_genes': 10000,
    'mlp_ratio': 4.0,

    # Evals
    'cheap_evals': ['batch_invariance', 'perturbation_detection', 'latent_space_health', 'reconstruction'],
    'full_evals': ['batch_invariance', 'perturbation_detection', 'latent_space_health', 'reconstruction',
                   'gene_embedding_pathways', 'essential_gene_prediction', 'embedding_consistency', 'cell_type_probing'],

    # Paths (MUST be Path objects)
    'data_root': Path('/Users/djemec/data/jepa/v0_6/'),
    'ref_dir': Path('/Users/djemec/data/jepa/reference_data/'),
    'pretraining_dir': Path('/Users/djemec/data/jepa/v0_6/pretraining/'),
}

In [ ]:
def create_study(cfg, phase=1):
    study_name = f'{cfg["study_name"]}_phase{phase}'
    sampler = TPESampler(
        n_startup_trials=cfg['n_startup_trials'],
        multivariate=cfg['multivariate'],
        constant_liar=cfg['constant_liar'],
        seed=cfg['seed']
    )
    pruner = MedianPruner(
        n_startup_trials=cfg['pruner_startup_trials'],
        n_warmup_steps=cfg['pruner_warmup_steps'],
        interval_steps=1
    )
    return optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=pruner,
        study_name=study_name,
        storage=f'sqlite:///{study_name}.db',
        load_if_exists=True
    )

In [ ]:
def create_hpo_model(params, cfg, device):
    '''Create BioJepa from HPO params with correct config field names.'''
    derived = derive_parameters(params)
    vicreg_w = params['vicreg_weight']

    model_cfg = BioJepaConfig(
        num_genes=cfg['num_genes'],
        n_layer=params['n_layer'],
        heads=params['heads'],
        embed_dim=params['embed_dim'],
        mlp_ratio=cfg['mlp_ratio'],
        n_pre_layer=derived['n_pre_layer'],
        mask_ratio=params['mask_ratio'],
        gaussian_scale=params['gaussian_scale'],
        film_linear_multiple=params['film_linear_multiple'],
        ema_momentum=params['ema_momentum'],
        sim_coeff=25.0 * vicreg_w,
        std_coeff=25.0 * vicreg_w,
        cov_coeff=1.0 * vicreg_w,
    )
    return create_model(model_cfg, device), model_cfg

In [ ]:
def objective(trial, cfg, phase=1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    params = {
        'embed_dim': trial.suggest_categorical('embed_dim', cfg['embed_dim_options']),
        'n_layer': trial.suggest_categorical('n_layer', cfg['n_layer_options']),
        'n_pre_layer_ratio': trial.suggest_float('n_pre_layer_ratio', *cfg['n_pre_layer_ratio_range']),
        'heads': trial.suggest_categorical('heads', cfg['heads_options']),
        'mask_ratio': trial.suggest_float('mask_ratio', *cfg['mask_ratio_range']),
        'gaussian_scale': trial.suggest_float('gaussian_scale', *cfg['gaussian_scale_range']),
        'ema_momentum': trial.suggest_float('ema_momentum', *cfg['ema_momentum_range']),
        'vicreg_weight': trial.suggest_float('vicreg_weight', *cfg['vicreg_weight_range']),
        'film_linear_multiple': trial.suggest_float('film_linear_multiple', *cfg['film_linear_multiple_range']),
        'lr': trial.suggest_float('lr', *cfg['lr_range'], log=True),
    }

    if not is_valid_config(params):
        raise optuna.TrialPruned('Invalid configuration')

    embed_dim = params['embed_dim']
    model, train_loader, eval_ctx = None, None, None

    try:
        model, model_cfg = create_hpo_model(params, cfg, device)

        if phase == 1:
            shard_paths = create_phase1_subsample(
                cfg['pretraining_dir'], cfg['phase1_fraction'], cfg['seed'], cfg['phase1_balance']
            )
            train_loader = SubsampledPretrainLoader(cfg['batch_size'], shard_paths, device)
        else:
            train_loader = PretrainLoader(cfg['batch_size'], 'train', cfg['pretraining_dir'], device)

        step_budget = compute_step_budget(embed_dim)
        report_step = 0
        current_step = 0

        eval_config = {
            'num_genes': cfg['num_genes'],
            'embed_dim': embed_dim,
            'n_layer': params['n_layer'],
            'heads': params['heads'],
            'batch_size': cfg['batch_size'],
        }

        model.enable_all_gradients()
        optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=0.05, fused=torch.cuda.is_available())
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=params['lr'], total_steps=step_budget, pct_start=0.05)

        model.train()
        while current_step < step_budget:
            steps_to_train = min(cfg['cheap_eval_interval'], step_budget - current_step)
            for _ in range(steps_to_train):
                batch = train_loader.next_batch()
                optimizer.zero_grad()
                loss = model.forward_pretrain(batch.x, batch.total)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                model.update_teacher()
                scheduler.step()
            current_step += steps_to_train

            is_full_eval = (current_step % cfg['full_eval_interval'] == 0)
            evals_to_run = cfg['full_evals'] if is_full_eval else cfg['cheap_evals']

            eval_ctx = EvalContext.from_trained_model(model, cfg['data_root'], cfg['ref_dir'], eval_config)
            eval_results = run_selected_evals(eval_ctx, evals_to_run)

            fail_checks = [
                check_vicreg_collapse(eval_results, embed_dim),
                check_perturbation_signal(eval_results),
                check_identity_shortcut_cheap(eval_results),
            ]
            for should_fail, reason in fail_checks:
                if should_fail:
                    trial.set_user_attr('fail_reason', reason)
                    raise optuna.TrialPruned(reason)

            if is_full_eval:
                fail_checks = [
                    check_identity_shortcut_full(eval_results),
                    check_cell_type_collapse(eval_results),
                ]
                for should_fail, reason in fail_checks:
                    if should_fail:
                        trial.set_user_attr('fail_reason', reason)
                        raise optuna.TrialPruned(reason)

                score = compute_objective(eval_results, embed_dim)
                trial.report(float(score), report_step)
                report_step += 1
                if trial.should_prune():
                    raise optuna.TrialPruned()

            del eval_ctx
            eval_ctx = None
            gc.collect()
            model.enable_all_gradients()
            model.train()

        model.eval()
        eval_ctx = EvalContext.from_trained_model(model, cfg['data_root'], cfg['ref_dir'], eval_config)
        final_results = run_selected_evals(eval_ctx, cfg['full_evals'])
        return float(compute_objective(final_results, embed_dim))

    finally:
        if eval_ctx is not None:
            del eval_ctx
        if model is not None:
            del model
        if train_loader is not None:
            del train_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
phase1_study = create_study(hpo_config, phase=1)
phase1_study.optimize(
    partial(objective, cfg=hpo_config, phase=1),
    n_trials=15,
    show_progress_bar=True
)

completed_trials = [t for t in phase1_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if not completed_trials:
    print('WARNING: No completed trials in Phase 1. Phase 2 will start without warm-start.')
    top_params = []
else:
    print(f'Phase 1 Best trial: {phase1_study.best_trial.number}')
    print(f'Phase 1 Best score: {phase1_study.best_value:.4f}')
    print(f'Phase 1 Best params: {phase1_study.best_params}')
    top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:5]
    top_params = [t.params for t in top_trials]
    print(f'Warm-starting Phase 2 with {len(top_params)} configs from Phase 1')

In [ ]:
phase2_study = create_study(hpo_config, phase=2)

for params in top_params:
    phase2_study.enqueue_trial(params)

phase2_study.optimize(
    partial(objective, cfg=hpo_config, phase=2),
    n_trials=25,
    show_progress_bar=True
)

completed_trials = [t for t in phase2_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if completed_trials:
    print(f'Phase 2 Best trial: {phase2_study.best_trial.number}')
    print(f'Phase 2 Best score: {phase2_study.best_value:.4f}')
    print(f'Phase 2 Best params: {phase2_study.best_params}')
else:
    print('WARNING: No completed trials in Phase 2.')

In [ ]:
df = phase2_study.trials_dataframe()
if df.empty:
    'No trials recorded.'
else:
    df.to_csv(f'{hpo_config["study_name"]}_results.csv', index=False)
    top_trials_df = df.nlargest(5, 'value')[['number', 'value', 'params_embed_dim', 'params_n_layer', 'params_mask_ratio']]
    top_trials_df if not top_trials_df.empty else 'No completed trials to display.'